In [1]:
import hashlib
from pathlib import Path
from tqdm.auto import tqdm

In [2]:
import numpy as np
import numexpr as ne
import healpy as hp
from scipy import interpolate
import pandas as pd
from sklearn.linear_model import LinearRegression
from pylab import cm
import matplotlib.pyplot as plt
from matplotlib.ticker import StrMethodFormatter, NullFormatter

from astropy import units as u
from astropy.coordinates import SkyCoord
from astropy.io import fits
from astropy.table import Table

from multiprocess import Pool
from pathlib import Path
import configparser
import nest_asyncio
import asyncio
import logging
import difflib
import shutil
import psutil
import time
import os
import gc

from tomographer.runtime_script import tomo_utils as utils

In [6]:
footprint_map = np.random.rand(12*2048**2)

In [7]:
import healpy as hp

hp.write_map('test_footprint_map.fits', footprint_map, overwrite=True, nest=True)

In [108]:
col = fits.Column(name='I', format='1E', array=footprint_map)
table_hdu = fits.BinTableHDU.from_columns([col])

hdul = fits.HDUList([fits.PrimaryHDU(), table_hdu])
hdul.writeto('test_footprint_map.fits', overwrite=True)

In [123]:
hrm, _ = hp.read_map('test_footprint_map.fits', nest=False, h=True)

In [109]:
tfm = fits.open('test_footprint_map.fits')[1].data

In [8]:
sfh = utils.safe_read_hmap('test_footprint_map.fits', 'NEST')

In [10]:
np.all(sfh == tfm['I']), tfm['I']

NameError: name 'tfm' is not defined

In [11]:
len(sfh)#, len(tfm)#, np.all(sfh == hrm)

50331648

In [118]:
import warnings
import logging
def safe_read_hmap(fname, in_ordering, out_ordering='NEST', out_nside=2048):
    """
    Read a HEALPix map of either the "proper" healpy format or a plain fits table. Data type should be float or int
    
    Args:
        fname (string): file name, including the full directory 
        in_ordering (string): HEALPix ordering of the input, "RING" or "NEST"
        out_ordering (string): HEALPix ordering of the output, "RING" or "NEST"
        out_nside (int): HEALPix nside of the output map
    Returns:
        array (numpy array): 1st column of the input with colname and unit stripped, properly nan-ed
    """

    if in_ordering=='NEST': nest=True
    else: nest=False
        
    # Handle HEALPix maps in both "proper" healpy map or plain fits table 
    data, hdr = hp.read_map(fname, nest=nest, h=True)
    hdr = dict(hdr)
    
    if 'ORDERING' in hdr:
        ftype = 'proper'
        hdr_in_ordering = hdr['ORDERING']
        in_nside = hdr['NSIDE']
    else:
        ftype = 'table' # In this case hp.read_map would have already taken only the 1st column and dropped the rest
    
    data = data.astype(float)
    data[data == hp.UNSEEN] = np.nan # Mask unseen value -1.6375e+30
    in_nside = hp.npix2nside(len(data))

    if (in_ordering!='NEST') or (in_nside!=2048):
        print('reordering')
        data = hp.ud_grade(data, out_nside, order_in=in_ordering, order_out=out_ordering) # includes reordering 
    
    if (ftype == 'proper'):
        hdr_norm = 'NEST' if hdr_in_ordering == 'NESTED' else hdr_in_ordering
        if in_ordering != hdr_norm:
            logging.warning("User specified ordering is different from the header.")
        
    return data

In [3]:
config = configparser.ConfigParser()
config.optionxform = str
config_filename = 'conf_YK_063026.ini'
config.read(config_filename)

['conf_YK_063026.ini']

In [16]:
combined_weighting

{'test_type': '',
 'test_data_file': '',
 'test_map_ordering': 'RING',
 'beam_fwhm_arcmin': '1.24',
 'footprint_definition': '',
 'test_random_file': '',
 'spatial_weight_map_file': '',
 'spatial_weight_map_ordering': 'RING',
 'per_source_weight_colname': '',
 'CSFD_cosmology_area': 'False',
 'SDSS_veto': 'False',
 'fsky_25_low_dust': 'False',
 'fsky_50_low_dust': 'False',
 'fsky_75_low_dust': 'False',
 'globular_clusters_veto': 'False',
 'LMC_SMC_veto': 'False',
 'nearby_galaxy_clusters_veto': 'False',
 'Planck_point_source_veto': 'False',
 'rec_cut': 'False',
 'rec_cut_coord': 'Galactic',
 'rec_include': 'True',
 'rec_min_lat': '-90',
 'rec_max_lat': '90',
 'rec_min_lon': '0',
 'rec_max_lon': '360',
 'additional_weight_map_file_1': '',
 'additional_weight_map_ordering_1': 'RING',
 'additional_weight_map_file_2': '',
 'additional_weight_map_ordering_2': 'RING',
 'N_prl': '4'}

In [21]:
weighting = config['Test Sample']
add_weighting = config['Optional Inputs']
combined_weighting = {**weighting, **add_weighting}

for weight in combined_weighting:
    if 'weight_map_file' in weight: 
        print(weight)
        wpath = combined_weighting.get(weight)
        print(wpath)

        ordering = weight.replace("_file", "_ordering")
        print(ordering, '\n')


spatial_weight_map_file

spatial_weight_map_ordering 

additional_weight_map_file_1

additional_weight_map_ordering_1 

additional_weight_map_file_2

additional_weight_map_ordering_2 



In [43]:
def verify_file_default(file_path, expected_sha256):
    sha256_hash = hashlib.sha256()
    
    # Read the file in chunks so it doesn't crash your RAM with large files
    try:
        with open(file_path, "rb") as f:
            for byte_block in iter(lambda: f.read(4096), b""):
                sha256_hash.update(byte_block)
        
        calculated_hash = sha256_hash.hexdigest()

        if calculated_hash != expected_sha256.lower():
            raise FileNotFoundError
            # self.DownloadError('The precalculated data files are corrupted. Please re-download.\n'
                                     # 'Did you install Git LFS to enable cloning large files?')
        return calculated_hash == expected_sha256.lower()
        
    except FileNotFoundError as e:
        raise FileNotFoundError('The precalculated data files are not found. Please re-download.\n'
                                'Did you install Git LFS to enable cloning large files?')

In [44]:
file_default.get('N_repeat_block_ID_1000BS_default_valid_pixels_flatten.npy')

NameError: name 'file_default' is not defined

In [45]:
file_default = {
        'bootstrapping/N_repeat_block_ID_1000BS_default_valid_pixels_flatten.npy': '45f8c76af6ba21fff17c6d0be1b7010ca385615749b9f230fc908933488c19d9',
        'bootstrapping/N_repeat_block_ID_100BS_default_valid_pixels_flatten.npy': '4bc53bd0188a9f2d80e3f2c957d11473572dcc6abb518f3ce6561dda79969188',
        'references/sdss_ref_all.fits': '10a66692c1b541be81aa346b34500c89b9a760a09c3a75ca720f6edd5e3d7928',
        'templates/NHI_ns2048_nest.fits': 'c582be84ac54db4be2752f7e90d0ae64723bf81001b4e7ef40d8d5a0f3030fd2',
        'templates/CSFD_ebv_ns2048_nest.fits': '1675a30f6eb81d238e4e6b27ac6d78567b4ce111d4500e41a9bde0e84535d869',
        'masks/LMC_SMC_veto.fits': '90ed3956278ee8ead25c7cb297ad91f8ff1f4fd3fac8f7e0128a13c3af9d0044',
        'masks/Planck_point_source_veto.fits': 'c5fcd76dee621465303c99879d2f17a9362d719a7ef6f88df9a1caf0e494dde2',
        'masks/fsky_75_low_dust.fits': '6712e73bb9cdcc41c2aea5e5239ba2f6c41cebdd4c13469edb6797aca4fad7df',
        'masks/CSFD_cosmology_area.fits': 'd7b1adda12779d9a5deacefca1f1615cf0f7b80111d065e5c32e116625f6348f',
        'masks/fsky_25_low_dust.fits': 'db80780898264ed144b90fc90f28f0cac49dca492ce025d39b08d7dc89f257f7',
        'masks/globular_clusters_veto.fits': '1a2ff0ca1ba4f6c613e482b6777b135e743d5861dc105f0610a3df1b421bccbc',
        'masks/nearby_galaxy_clusters_veto.fits': '1b2714b2001eb2c0e48d9def86311e9b2c4577d90d1139a5153fac8fbada012e',
        'masks/fsky_50_low_dust.fits': 'e9345a95ffb28ba1c523b5f0b9cabbf7d3b62628aa8a470e40a15c1a18681866',
        'masks/SDSS_veto.fits': '201e23c9ca5576263c233c7a7bdaa0cac4e54212097909ebc93c6c84d5585f4b',
        'references/40log1pzbins/sdss_lss_plus_data_z_maps.pddf.parquet': '4bd31e560dd7317d997912c162f12772423c1a1385b5c87b12b237e9bbf70020',
        'references/40log1pzbins/b_ref_sdss_lss_plus_0.5-10Mpch_gamma-0.8.fits': '634baa5637d658a35a7ae83acf30ce7ea5c0539065ae4e7ccc86b1c205d18d05',
        'references/footprint/sdss_ref_footprint_nest2048.fits': '095ec6ef2c4fb4f9baa2e40311b50af9b158e45eca42f2bed3720dc09575ee40',
        'references/footprint/valid_pixel_ids_nest2048.npz': 'b9245924c6aa6abc9e443c08ff9000759cb0685ad6a900b953dca740dc37b04b',
        'references/footprint/valid_pixels_nest2048.fits': 'f627847d824f934b0a99ea9a91774d6a947af3c7255cf754b8597992e79f15d6',
        'cosmology/planck_2018/wmbar_prp_0.5-10Mpch_gamma-0.8_15.0degtmax_40log1pzbins_multi_beam_filtering.fits': 'b115309d3e094ebd1f679353be49f05d301a9baa60c8c102ceabc259a1996c3d',
        'cosmology/planck_2018/cosmology.txt': 'ad76bf486afc97642cc667f0ead76fb34cf343e2764fd231c8e171e03396fe36',
        'activation_maps/40log1pzbins/sdss_lss_plus_data_0.5-10Mpch_gamma-0.8.pddf.parquet': 'bc84fd892df86d2149dbfc8c820f44d911bfb6afe3bb3d4cdbba422e5e28c580',
        'activation_maps/40log1pzbins/sdss_lss_plus_rand_0.5-10Mpch_gamma-0.8.pddf.parquet': '3c86e1f232c4cb0b4315722c94e44c7c897bb087f2e1769d2f0db3081e273c2b',
        }

In [51]:
def calculate_sha256(file_path: Path) -> str:
    """Streams a file in chunks to calculate its SHA-256 hash without overloading RAM."""
    sha256_hash = hashlib.sha256()
    try:
        with open(file_path, "rb") as f:
            # Read in 64KB chunks
            for byte_block in iter(lambda: f.read(65536), b""):
                sha256_hash.update(byte_block)
        return sha256_hash.hexdigest()
    except Exception as e:
        return f"ERROR: Could not read file ({e})"

In [52]:
root_path = Path('tomographer/precalculated_data/')

for item in root_path.rglob("*"):
    if item.is_file():  # Skip directories, only hash files
        file_hash = calculate_sha256(item)
        
        # Get the path relative to your starting directory for clean output
        relative_path = item.relative_to(root_path)
        
        print(f"'{relative_path}': '{file_hash}',")

'.DS_Store': '46b61cacbd6f87b5f987a2ca95a7afaa02b865860689819b9dd5594b67b76394',
'bootstrapping/N_repeat_block_ID_1000BS_default_valid_pixels_flatten.npy': '8f79da4e509d8ff6783a8234df270ddb82473dbb8445a7d9df54ea1781860b9d',
'bootstrapping/N_repeat_block_ID_100BS_default_valid_pixels_flatten.npy': 'f43fd498edd94c3c48dee020ef6544cd491633b1d5512e78c3d072cc53079f63',
'references/.DS_Store': '43031008d07ea2babe25603efac1556760fb2fda38b28b8ea12d460fc2840dc8',
'references/sdss_ref_all.fits': '10a66692c1b541be81aa346b34500c89b9a760a09c3a75ca720f6edd5e3d7928',
'templates/NHI_ns2048_nest.fits': 'c582be84ac54db4be2752f7e90d0ae64723bf81001b4e7ef40d8d5a0f3030fd2',
'templates/CSFD_ebv_ns2048_nest.fits': '1675a30f6eb81d238e4e6b27ac6d78567b4ce111d4500e41a9bde0e84535d869',
'cosmology/.DS_Store': '36eca623588191bb0419fef74ed26091c7372366d5ab1b64c974f209d5317e5f',
'masks/LMC_SMC_veto.fits': '90ed3956278ee8ead25c7cb297ad91f8ff1f4fd3fac8f7e0128a13c3af9d0044',
'masks/Planck_point_source_veto.fits': 'c5fcd76

In [34]:
import sys
# Insert the directory path at index 0 to give it search priority
sys.path.insert(1, 'tomographer/runtime_script/')
import tomo_utils as utils

In [86]:
import importlib
importlib.reload(utils)

<module 'tomo_utils' from '/Users/yuvoon/Documents/clustering_redshift/tomographer/tomographer/runtime_script/tomo_utils.py'>

In [87]:
filepath_handler = utils.FilePathHandler()
filepath_handler.check_file_default()

In [64]:
file_default.keys()

dict_keys(['bootstrapping/N_repeat_block_ID_1000BS_default_valid_pixels_flatten.npy', 'bootstrapping/N_repeat_block_ID_100BS_default_valid_pixels_flatten.npy', 'references/sdss_ref_all.fits', 'templates/NHI_ns2048_nest.fits', 'templates/CSFD_ebv_ns2048_nest.fits', 'masks/LMC_SMC_veto.fits', 'masks/Planck_point_source_veto.fits', 'masks/fsky_75_low_dust.fits', 'masks/CSFD_cosmology_area.fits', 'masks/fsky_25_low_dust.fits', 'masks/globular_clusters_veto.fits', 'masks/nearby_galaxy_clusters_veto.fits', 'masks/fsky_50_low_dust.fits', 'masks/SDSS_veto.fits', 'references/40log1pzbins/sdss_lss_plus_data_z_maps.pddf.parquet', 'references/40log1pzbins/b_ref_sdss_lss_plus_0.5-10Mpch_gamma-0.8.fits', 'references/footprint/sdss_ref_footprint_nest2048.fits', 'references/footprint/valid_pixel_ids_nest2048.npz', 'references/footprint/valid_pixels_nest2048.fits', 'cosmology/planck_2018/wmbar_prp_0.5-10Mpch_gamma-0.8_15.0degtmax_40log1pzbins_multi_beam_filtering.fits', 'cosmology/planck_2018/cosmology

In [72]:
for relative_path in file_default.keys():
    absolute_path = Path('tomographer/precalculated_data')/relative_path
    file_default_indict = file_default.get(str(relative_path))
    print(relative_path)
    file_hash = verify_file_default(absolute_path, file_default_indict)

bootstrapping/N_repeat_block_ID_1000BS_default_valid_pixels_flatten.npy
bootstrapping/N_repeat_block_ID_100BS_default_valid_pixels_flatten.npy
references/sdss_ref_all.fits


FileNotFoundError: The precalculated data files are not found. Please re-download.
Did you install Git LFS to enable cloning large files?

In [26]:
from pathlib import Path
from astropy.io import fits
from astropy.table import Table


In [38]:
measurement_file = ('/Users/yuvoon/Documents/clustering_redshift/dev_v08/tomographer/runtime_test/test_single_lowz_40bins_rebin38.fits')
bin_number = 20


In [39]:
w = Table.read(measurement_file)

In [35]:
rebinned_z_edges, rebinned_z_ctrs = utils.z_binning_log1pz(0., 4.2, bin_number)
utils.z_rebin_tomo_out(w, rebinned_z_edges, rebinned_z_ctrs)

z,z_eff,z_low,z_up,dz,w_rr,w_rr_err,w_tr,DD_tr,DR_tr,RD_tr,RR_tr,w_tr_err,w_tr_BS,w_m_ref_auto,w_m_cross,b_r_final,b_r_err,b_r_SNR,dNdz_b,dNdz_b_err,dNdz_b_BS
float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64[32],float64,float64,float64,float64,float64,float64,float64,float64[32]
0.04207765513895101,0.05780067987193319,0.0,0.08592583933989428,0.04360189150450581,0.36273944972088484,0.039191194541022045,0.006719909066121114,2006077839845.911,2047598270611.2434,3053888083070.745,3111332867327.697,0.005964990327816554,0.014041940457172645 .. 0.015142287661097657,0.012868520620023817,0.007155743384036263,1.1117344867930503,0.060609897237710694,15.509829942542414,0.9544498607400652,0.7228773956179612,1.5459047605963647 .. 1.8436272525814854
0.1316190523141143,0.1497557042727575,0.08592583933989428,0.17923492854605416,0.04739222088650675,0.42802156957113824,0.024546998603617826,0.005261525934274042,1485584311761.797,1500010774872.2512,2252950795490.354,2278080119998.97,0.004283631986524237,0.010373652384629973 .. 0.007847245113900848,0.01101315126885027,0.008613573064470047,1.3443840145983523,0.03875781754344046,30.563706400480143,0.47479233119353464,0.3678211329815093,0.9155178752499491 .. 0.6932054089718955
0.22885436919722046,0.2445221572847916,0.17923492854605416,0.2805616795602941,0.05129843562159053,0.5765102905481587,0.018439527410227686,-0.0020443295660857726,1110142618893.5857,1099641613925.3623,1689057170525.0732,1675141820649.8286,0.0034285011377189188,-0.0008787144944616385 .. 0.001989460845102632,0.00952874062295895,0.008170172864976627,1.8678279159649693,0.030067759143097887,50.85545049388325,-0.13041192200881938,0.22364559050161398,-0.05300878231189933 .. 0.13731226161550084
0.334444712296988,0.3342913888042933,0.2805616795602941,0.3905950167030172,0.05499866796936152,0.5152216158815739,0.013870592632556094,0.0024255565866065742,1199124992286.0496,1201871251289.8484,1825919779102.6328,1832717142061.798,0.0024150906607419845,0.002004157157546173 .. 0.006415112383401682,0.008416160679542446,0.0075191409735342,1.8737491825479697,0.02464934715871453,56.7646910723236,0.16856136040854514,0.17056667682423182,0.1315848035395318 .. 0.4492260083320812
0.44910799425379055,0.4440778258625683,0.3905950167030172,0.5100830606950983,0.05952399718153476,0.44147258858347715,0.00953889556132903,-0.0007063792907037781,1280319075115.1763,1267966000045.1777,1946799298093.9487,1932813865715.8896,0.0021875480328961,-0.0025522170829049362 .. -0.002778283379383956,0.0073353066124393716,0.006742125567214992,1.9413464865799046,0.0207502784541884,67.01326625240377,-0.054599123944621616,0.16622655873590111,-0.20295932008610634 .. -0.22722410246312374
0.5736238149541981,0.5714055700573726,0.5100830606950983,0.6398382151582813,0.06477236825326398,0.35167151082744574,0.006325570136963745,0.000825925027671504,1688565499287.404,1685731964210.9185,2566382578650.911,2565891152029.043,0.0018354304011196573,0.0014266104407475836 .. 0.00016900952738912035,0.006367480813908224,0.005965219650590102,1.9382200506288378,0.01759620704865768,80.06891654086625,0.07127705528949774,0.15806271804458027,0.1243994529331438 .. 0.017080211682377916
0.7088387620593846,0.700283212042007,0.6398382151582813,0.7807426901773911,0.07008462515581293,0.25953413636684164,0.0067038280046091536,0.012208576841494425,1361562048687.321,1338526560390.4897,1985259363414.037,1989508280170.748,0.001639642982642905,0.009576813877791673 .. 0.012872086869391081,0.005608753273812122,0.005318562296961694,1.8576378746445967,0.025469077095764802,58.97239863071801,1.3626080686909647,0.1642022251784665,1.1424487885572159 .. 1.4403652643007967
0.8556721669858831,0.8661021076253651,0.7807426901773911,0.9337545004792649,0.0769195777439488,0.15560826380485196,0.008866223789800969,0.05394103400208177,1040940360095.715,960049133355.3048,1375291336187.0452,1379727896514.8318,0.0024803558375889392,0.05488935083168602 .. 0.05246113362156947,0.0048

In [49]:
Path('out').stem

'out'

In [5]:
path = Path('./newtest/newtest_ddz.fits')

In [8]:
if not path.parent.exists():
    mkdir(path.parent)

NameError: name 'mkdir' is not defined